In [1]:
import os
import torch as t
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torchvision import datasets, transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torchvision.transforms import Compose, ToTensor, PILToTensor, Resize
from tqdm import tqdm
import numpy as np
import optuna as opt
from ultralytics import YOLO
from ultralytics.engine import model as Model
import albumentations as A

/home/var-roman/anaconda3/envs/Volleyball_diploma/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class YOLODataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        self.imgs = os.listdir(img_dir)
        self.labels = os.listdir(label_dir)

        self.img_names = [f for f in self.imgs if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        # Reading images
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        label_name = os.path.splitext(img_path)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_name)

        boxes = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as r:
                for line in r.readlines():
                    parts = line.strip().split()
                    class_id = int(parts[0])
                    x_c, y_c, w, h = map(float, parts[1:])
                    boxes.append([class_id, x_c, y_c, w, h])

        boxes = t.tensor(boxes, dtype=t.float32) if len(boxes) > 0 else t.zeros((0, 5))

        if self.transform:
            image = self.transform(image)

        return image, boxes

def yolo_collate_fn(batch):
    """
    Unite list of tuples(image, boxes) into the batch
    images dimensions: [batch_size, 3, height, width]
    targets dimensions: [batch_size, num_of_boxes, 5] (stays the same)
    """
    images, targets = [], []
    for i, (image, boxes) in enumerate(batch):
        images.append(img)
        if len(boxes) > 0:
            batch_idx = t.full((boxes.shape[0], 1), i)
            batch_with_idx = t.cat([batch_idx, boxes], dim=1)
            targets.append(batch_with_idx)

    images = t.stack(images, dim=0)
    if len(targets) > 0:
        targets = t.cat(targets, dim=0)
    else:
        targets = t.zeros((0, 6))

    return images, targets

In [3]:
# For volleyball_detection
dir_1 = '/home/var-roman/Desktop/my_projects/diploma_project/data/volleyball_detection.yolo26'
# For vv_dataset_only_ball
dir_2 = '/home/var-roman/Desktop/my_projects/diploma_project/data/vv_dataset_only_ball.v2i.yolo26'

In [4]:
transform = Compose([
    Resize((640, 640)),
    ToTensor(),
])

train_dataset_1 = YOLODataset(
    img_dir=dir_1+'/train/images',
    label_dir=dir_1+'/train/labels',
    transform=transform
)

# train_dataset_2 = YOLODataset(
#     img_dir=dir_2+'/train/images',
#     label_dir=dir_2+'/train/labels',
#     transform=transform
# )


# main_train_dataset = ConcatDataset([train_dataset_1, train_dataset_2])

# train_loader = DataLoader(
#     main_train_dataset,
#     batch_size=16,
#     shuffle=True,
#     collate_fn=yolo_collate_fn,
#     num_workers=8,
#     pin_memory=True
# )

train_loader_1 = DataLoader(
    train_dataset_1,
    batch_size=16,
    shuffle=True,
    collate_fn=yolo_collate_fn,
    num_workers=8,
    pin_memory=True
)

# train_loader_2 = DataLoader(
#     train_dataset_2,
#     batch_size=16,
#     shuffle=True,
#     collate_fn=yolo_collate_fn,
#     num_workers=8,
#     pin_memory=True
# )

In [8]:
# main_model = YOLO('/home/var-roman/Desktop/my_projects/diploma_project/models/yolo26s.pt')
main_model = YOLO('yolo26m.pt')

custom_transforms = [
    A.MotionBlur(blur_limit=(7, 15), p=0.4),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
    A.CLAHE(clip_limit=4.0, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.4),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.3),
]

In [6]:
main_model.train(data='/home/var-roman/Desktop/my_projects/diploma_project/data/main_data.yaml',
                 epochs=25, device='0', imgsz=640, augmentations=custom_transforms)

Ultralytics 8.4.38 🚀 Python-3.11.14 torch-2.4.0 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5762MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[MotionBlur(p=0.4, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(7, 15), direction_range=(-1.0, 1.0)), GaussNoise(p=0.2, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.01, 0.05)), CLAHE(p=0.3, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)), RandomBrightnessContrast(p=0.4, brightness_by_max=True, brightness_limit=(-0.25, 0.25), contrast_limit=(-0.25, 0.25), ensure_safe_range=False), HueSaturationValue(p=0.3, hue_shift_limit=(-15.0, 15.0), sat_shift_limit=(-30.0, 30.0), val_shift_limit=(-20.0, 20.0))], auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/var-roman/Desktop/my_projects/diplo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78efd47af1d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
metrics = main_model.val(data='/home/var-roman/Desktop/my_projects/diploma_project/data/volleyball_detection.yolo26/data.yaml', imgsz=640)

Ultralytics 8.4.38 🚀 Python-3.11.14 torch-2.4.0 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5762MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5193.1±911.8 MB/s, size: 972.2 KB)
val: Scanning /home/var-roman/Desktop/my_projects/diploma_project/data/volleyball_detection.yolo26/valid/labels... 109 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 109/109 3.2Kit/s 0.0s
val: New cache created: /home/var-roman/Desktop/my_projects/diploma_project/data/volleyball_detection.yolo26/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 3.1it/s 2.3s0.3ss
                   all        109        152       0.94      0.954      0.954       0.73
Speed: 2.1ms preprocess, 10.8ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/runs/detect/

In [8]:
print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")

mAP50: 0.954235419691978
mAP50-95: 0.7298079283356084


In [9]:
main_model.save('/home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/models/ball_detec_v1.pt')

In [14]:
# To load model
main_model = YOLO('/home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/models/ball_detec_v1.pt')

In [15]:
results = main_model.predict(
    source='/home/var-roman/Desktop/my_projects/diploma_project/data/videos/video_2026-04-17_17-20-41.mp4',
    imgsz=640,
    conf=0.65,
    iou=0.45,
    save=True,
    show=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1087) /home/var-roman/Desktop/my_projects/diploma_project/data/videos/video_2026-04-17_17-20-41.mp4: 384x640 (no detections), 19.1ms
video 1/1 (frame 2/1087) /home/var-roman/Desktop/my_projects/diploma_project/data/videos/video_2026-04-17_17-20-41.mp4: 384x640 (no detections), 11.0ms
video 1/1 (frame 3/1087) /home/var-roman/Desktop/my_projects/diploma_project/data/videos/video_2026-04-17_17-20-41.mp4: 384x640 (no detections), 11.8ms
vi

In [19]:
# # Експорт для відеокарт NVIDIA (забезпечує максимальний FPS)
# main_model.export(format='engine', dynamic=True, half=True) # TensorRT
#
# # Експорт для CPU або кросплатформного використання
# main_model.export(format='onnx')
#
# # Тестування швидкості експортованої моделі
# trt_model = YOLO('runs/detect/custom_training/weights/best.engine')
# trt_model.predict(source='test.mp4', device=0)

WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.38 🚀 Python-3.11.14 torch-2.4.0 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5762MiB)
WARNING ⚠️ 'dynamic=True' model with 'format=engine' requires max batch size, i.e. 'batch=16'

PyTorch: starting from '/home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/runs/detect/train5/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (42.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/17.6 MB ? eta -:--:--
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/17.6 MB 9.5 MB/s eta 0:00:02
   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/17.6 MB 16.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━ 8.7/17.6 MB 18.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━ 11.0/17.6 MB 15.5 MB/s eta 

/home/var-roman/anaconda3/envs/Volleyball_diploma/lib/python3.11/site-packages/torch/onnx/symbolic_opset9.py:5715: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 22.8s, saved as '/home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/runs/detect/train5/weights/best.onnx' (78.6 MB)
requirements: Ultralytics requirement ['tensorrt-cu12>=7.0.0,!=10.1.0'] not found, attempting AutoUpdate...
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━

/home/var-roman/anaconda3/envs/Volleyball_diploma/lib/python3.11/site-packages/torch/onnx/symbolic_opset9.py:5715: UserWarning: Exporting aten::index operator of advanced indexing in opset 19 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 3.9s, saved as '/home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/runs/detect/train5/weights/best.onnx' (77.9 MB)

Export complete (5.4s)
Results saved to /home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/runs/detect/train5/weights
Predict:         yolo predict task=detect model=/home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/runs/detect/train5/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/home/var-roman/Desktop/my_projects/diploma_project/parsing_datasets_scripts/runs/detect/train5/weights/best.onnx imgsz=640 data=/home/var-roman/Desktop/my_projects/diploma_project/data/volleyball_detection.yolo26/data.yaml  
Visualize:       https://netron.app


FileNotFoundError: 'runs/detect/custom_training/weights/best.engine' does not exist